In [44]:
import re
import json
import time
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup


# =========================================================
# INPUT
# =========================================================
CIBC_CARDS = [
    {
        "card_id": "cibc_aventura_visa_infinite_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/aventura-visa-infinite-card.html"
    },
    {
        "card_id": "cibc_aeroplan_visa_infinite_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/aeroplan-visa-infinite-card.html"
    },
    {
        "card_id": "cibc_aeroplan_visa_infinite_privilege_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/aeroplan-visa-infinite-privilege-card.html"
    },
    {
        "card_id": "cibc_aventura_gold_visa_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/aventura-gold-visa-card.html"
    },
    {
        "card_id": "cibc_aventura_visa_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/aventura-visa-card.html"
    },

    {
        "card_id": "cibc_aventura_visa_infinite_privilege_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/aventura-visa-infinite-privilege-card.html"
    },



    {
        "card_id": "cibc_aaeroplan_visa_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/aeroplan-visa-card.html"
    },
    {
        "card_id": "cibc_adapta_mastercard_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/adapta-mastercard.html"
    },

    {
        "card_id": "cibc_dividend_visa_infinite_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/dividend-visa-infinite-card.html"
    },

    {
        "card_id": "cibc_dividend_visa_platinum_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/dividend-visa-platinum-card.html"
    },
    {
        "card_id": "cibc_dividend_visa_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/dividend-visa-card.html"
    },
    {
        "card_id": "cibc_costco_mastercard_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/costco-mastercard.html"
    },

    {
        "card_id": "cibc_adapta_mastercard_students_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/adapta-mastercard-for-students.html"
    },
    {
        "card_id": "cibc_dividend_visa_students_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/dividend-visa-for-students.html"
    },
    {
        "card_id": "cibc_aventura_visa_students_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/aventura-visa-for-students.html"
    },
    {
        "card_id": "cibc_aeroplan_visa_students_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/aeroplan-visa-for-students.html"
    },
    {
        "card_id": "cibc_usdollar_aventura_gold_visa_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/us-dollar-aventura-gold-visa-card.html"
    },

    {
        "card_id": "cibc_select-visa_ca",
        "url": "https://www.cibc.com/en/personal-banking/credit-cards/all-credit-cards/select-visa-card.html"
    },
    {
        "card_id": "cibc_aventura_visa_business_ca",
        "url": "https://www.cibc.com/en/business/credit-cards/aventura-visa.html"
    },
    {
        "card_id": "cibc_aeroplan_visa_business_ca",
        "url": "https://www.cibc.com/en/business/credit-cards/aeroplan-visa-business.html"
    },
    {
        "card_id": "cibc_costco_mastercard_business_ca",
        "url": "https://www.cibc.com/en/business/credit-cards/costco-mastercard-business.html"
    },
    {
        "card_id": "cibc_aventura_plus_visa_ca",
        "url": "https://www.cibc.com/en/business/credit-cards/aventura-plus-visa.html"
    },
    {
        "card_id": "cibc_aeroplan_visa_business_plus_ca",
        "url": "https://www.cibc.com/en/business/credit-cards/aeroplan-visa-business-plus.html"
    },
    {
        "card_id": "cibc_bizline_visa_ca",
        "url": "https://www.cibc.com/en/business/credit-cards/bizline-visa.html"
    },
    {
        "card_id": "cibc_corporate_classic_plus_visa_ca",
        "url": "https://www.cibc.com/en/business/credit-cards/corporate-classic-plus-visa.html"
    }

]

DATA_DIR = Path("./credit-card-rag/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "cibc_cards_min.csv"
SNIPPETS_PATH = DATA_DIR / "cibc_card_snippets.jsonl"
DEBUG_DIR = DATA_DIR / "cibc_debug_pages"
DEBUG_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# FETCH
# =========================================================
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-CA,en;q=0.9",
}

session = requests.Session()
session.headers.update(HEADERS)


def fetch_page_text(url: str, sleep_sec: float = 1.5) -> str:
    """
    Fetch page HTML and convert to line-preserving plain text.
    """
    response = session.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "lxml")
    text = soup.get_text("\n", strip=True)
    time.sleep(sleep_sec)
    return normalize_text(text)


# =========================================================
# TEXT HELPERS
# =========================================================
def normalize_text(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", " ")

    # normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # trim each line
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.split("\n")]

    # drop empty spam repeats
    cleaned = []
    prev = None
    for line in lines:
        if not line:
            continue
        if line == prev:
            continue
        cleaned.append(line)
        prev = line

    return "\n".join(cleaned).strip()


def to_one_line(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def slug_from_url(url: str) -> str:
    return url.rstrip("/").split("/")[-1].replace(".html", "")


def infer_family(card_id: str, card_name: str) -> str:
    txt = f"{card_id} {card_name}".lower()
    if "aeroplan" in txt:
        return "aeroplan"
    if "aventura" in txt:
        return "aventura"
    return "generic"


def infer_network(card_name: str, url: str) -> str:
    txt = f"{card_name} {url}".lower()
    if "visa" in txt:
        return "Visa"
    if "mastercard" in txt:
        return "Mastercard"
    return ""


def infer_one_liner(family: str) -> str:
    if family == "aventura":
        return "Travel rewards credit card designed for flexible Aventura points and travel perks."
    if family == "aeroplan":
        return "Travel rewards credit card focused on Aeroplan points and Air Canada-related benefits."
    return "CIBC credit card with rewards, benefits, and cardholder perks."


def infer_best_for(family: str, text: str) -> str:
    t = text.lower()
    tags = []

    if family in {"aventura", "aeroplan"}:
        tags.append("travel")
    if "aeroplan" in t:
        tags.append("air_travel")
    if "grocery" in t or "groceries" in t:
        tags.append("groceries")
    if "gas" in t:
        tags.append("gas")
    if "drug store" in t or "drug stores" in t:
        tags.append("drugstores")
    if "airport" in t or "lounge" in t:
        tags.append("airport_benefits")

    # de-duplicate keeping order
    return ",".join(dict.fromkeys(tags))


def save_debug_text(card_id: str, text: str) -> None:
    path = DEBUG_DIR / f"{card_id}.txt"
    path.write_text(text, encoding="utf-8")


# =========================================================
# LINE / SECTION HELPERS
# =========================================================
def lines_of(text: str) -> list[str]:
    return [line.strip() for line in text.split("\n") if line.strip()]


def find_all_indexes(lines: list[str], pattern: str) -> list[int]:
    rx = re.compile(pattern, re.IGNORECASE)
    return [i for i, line in enumerate(lines) if rx.search(line)]


def first_index(lines: list[str], patterns: list[str], start: int = 0) -> int:
    for i in range(start, len(lines)):
        for pattern in patterns:
            if re.search(pattern, lines[i], re.IGNORECASE):
                return i
    return -1


def next_nonempty_value(lines: list[str], label_patterns: list[str], lookahead: int = 8) -> str:
    idx = first_index(lines, label_patterns)
    if idx == -1:
        return ""

    for j in range(idx + 1, min(len(lines), idx + 1 + lookahead)):
        candidate = lines[j].strip()
        if not candidate:
            continue
        if candidate.lower().startswith("opens in"):
            continue
        if candidate in {"* * *", "Apply now", "Learn more"}:
            continue
        if re.match(r"^#+\s*", candidate):
            continue
        return candidate

    return ""


def section_between(
    lines: list[str],
    start_patterns: list[str],
    end_patterns: list[str],
    start_pos: int = 0,
    include_start: bool = True
) -> str:
    start_idx = first_index(lines, start_patterns, start=start_pos)
    if start_idx == -1:
        return ""

    end_idx = len(lines)
    for i in range(start_idx + 1, len(lines)):
        for pattern in end_patterns:
            if re.search(pattern, lines[i], re.IGNORECASE):
                end_idx = i
                return "\n".join(lines[start_idx:end_idx] if include_start else lines[start_idx + 1:end_idx]).strip()

    return "\n".join(lines[start_idx:end_idx] if include_start else lines[start_idx + 1:end_idx]).strip()


def extract_main_card_name(lines: list[str], fallback_slug: str = "") -> str:
    """
    Pick the main card heading near the main product block, not the promo banner near the top.
    """
    fee_idx = first_index(lines, [r"^Fees and interest rates$"])
    candidates = []

    for i, line in enumerate(lines):
        if re.search(r"^CIBC .* Card\.?$", line, re.IGNORECASE):
            candidates.append((i, line.rstrip(".")))

    if not candidates:
        for i, line in enumerate(lines):
            if re.search(r"^CIBC .* Card$", line, re.IGNORECASE):
                candidates.append((i, line))

    if fee_idx != -1:
        before_fee = [c for c in candidates if c[0] < fee_idx]
        if before_fee:
            chosen = before_fee[-1][1]
            return chosen.replace("*", "").strip()

    if candidates:
        return candidates[-1][1].replace("*", "").strip()

    if fallback_slug:
        return fallback_slug.replace("-", " ").title()

    return ""


def crop_to_main_product_block(lines: list[str], card_name: str) -> list[str]:
    """
    Remove global site header/footer noise and keep only the main product block.
    """
    start_idx = 0
    end_idx = len(lines)

    if card_name:
        idxs = find_all_indexes(lines, re.escape(card_name))
        if idxs:
            # choose the one closest before fees section if possible
            fee_idx = first_index(lines, [r"^Fees and interest rates$"])
            if fee_idx != -1:
                before_fee = [i for i in idxs if i <= fee_idx]
                if before_fee:
                    start_idx = max(0, before_fee[-1] - 2)
                else:
                    start_idx = max(0, idxs[-1] - 2)
            else:
                start_idx = max(0, idxs[-1] - 2)

    for marker in [r"^Questions\?$", r"^Terms and conditions$", r"^Feedback$", r"^Find a branch or ATM$"]:
        idx = first_index(lines, [marker], start=start_idx)
        if idx != -1:
            end_idx = min(end_idx, idx)
            break

    return lines[start_idx:end_idx]


def make_snippet(card_id: str, section: str, text: str):
    text = to_one_line(text)
    if not text:
        return None
    return {
        "chunk_id": f"{card_id}_{section}",
        "card_id": card_id,
        "section": section,
        "text": text
    }


# =========================================================
# FIELD EXTRACTORS
# =========================================================
def extract_income_requirement(lines: list[str]) -> str:
    for line in lines:
        if "minimum annual income" in line.lower():
            return line
    return ""


def extract_annual_fee(lines: list[str]) -> str:
    value = next_nonempty_value(lines, [r"^Annual fee$"])
    m = re.search(r"\$([0-9,]+(?:\.[0-9]{1,2})?)", value)
    return m.group(1).replace(",", "") if m else ""


def extract_annual_fee_full(lines: list[str]) -> str:
    value = next_nonempty_value(lines, [r"^Annual fee$"])
    return value


def extract_additional_card_fee(lines: list[str]) -> str:
    idx = first_index(lines, [r"^Annual fee$"])
    if idx == -1:
        return ""
    for j in range(idx + 2, min(len(lines), idx + 8)):
        if "$" in lines[j]:
            return lines[j]
    return ""


def extract_purchase_interest_rate(lines: list[str]) -> str:
    value = next_nonempty_value(lines, [r"^Purchase interest rate"])
    if re.fullmatch(r"[0-9]{1,2}\.[0-9]{2}%", value):
        return value
    return ""


def extract_cash_interest_rate(lines: list[str]) -> str:
    value = next_nonempty_value(lines, [r"^Cash interest rate"])
    if re.fullmatch(r"[0-9]{1,2}\.[0-9]{2}%", value):
        return value
    return ""


def extract_welcome_bonus_summary(lines: list[str]) -> str:
    block = section_between(
        lines,
        start_patterns=[r"^Digital exclusive offer$", r"^Welcome Offer$"],
        end_patterns=[r"^Show more offer details", r"^Apply now$", r"^Fees and interest rates$"]
    )
    if not block:
        return ""

    block_lines = lines_of(block)
    for line in block_lines[1:6]:
        if any(word in line.lower() for word in ["earn up to", "apply and earn up to", "get a total of up to"]):
            return line
    return block_lines[1] if len(block_lines) > 1 else block_lines[0]


def extract_offer_block(lines: list[str]) -> str:
    return section_between(
        lines,
        start_patterns=[r"^Digital exclusive offer$", r"^Welcome Offer$"],
        end_patterns=[r"^Fees and interest rates$"]
    )


def extract_overview_block(lines: list[str], card_name: str) -> str:
    start_patterns = []
    if card_name:
        start_patterns.append(rf"^{re.escape(card_name)}\.?$")
    start_patterns.extend([
        r"^Travel rewards cards$",
        r"^Air Canada Aeroplan credit cards$",
    ])

    return section_between(
        lines,
        start_patterns=start_patterns,
        end_patterns=[r"^Fees and interest rates$"]
    )


def extract_rewards_block(lines: list[str], family: str) -> str:
    if family == "aventura":
        start_patterns = [r"^Earn more Aventura travel points$", r"^Earn more Aventura points$"]
    elif family == "aeroplan":
        start_patterns = [r"^Earn Aeroplan points$", r"^Earn more Aeroplan points$", r"^Earn points faster$"]
    else:
        start_patterns = [r"^Earn points$", r"^Earn rewards$"]

    return section_between(
        lines,
        start_patterns=start_patterns,
        end_patterns=[
            r"^Discover the amazing features",
            r"^Your top .* card benefits$",
            r"^More to love about this card$",
            r"^Protecting your card$",
        ]
    )


def extract_benefits_block(lines: list[str], family: str) -> str:
    start_patterns = [r"^More to love about this card$"]
    if family == "aventura":
        start_patterns = [r"^Your top Aventura card benefits$", r"^More to love about this card$"]
    elif family == "aeroplan":
        start_patterns = [r"^Your top Aeroplan card benefits$", r"^More to love about this card$"]

    return section_between(
        lines,
        start_patterns=start_patterns,
        end_patterns=[
            r"^Calculate how many",
            r"^Protecting your card$",
            r"^Insurance benefits$",
            r"^Documents$",
            r"^CIBC Pace It",
        ]
    )


def extract_insurance_block(lines: list[str]) -> str:
    return section_between(
        lines,
        start_patterns=[r"^Valuable insurance included with your card"],
        end_patterns=[
            r"^Optional insurance for your card$",
            r"^Documents$",
            r"^CIBC Pace It",
        ]
    )


def extract_documents_block(lines: list[str]) -> str:
    return section_between(
        lines,
        start_patterns=[r"^Documents$"],
        end_patterns=[r"^CIBC Pace It", r"^Ready to start", r"^CIBC [A-Z ]+ CARD$"]
    )


# =========================================================
# MAIN SCRAPER
# =========================================================
def scrape_cibc_card(card_cfg: dict) -> tuple[dict, list[dict]]:
    card_id = card_cfg["card_id"]
    url = card_cfg["url"]

    raw_text = fetch_page_text(url)
    save_debug_text(card_id, raw_text)

    raw_lines = lines_of(raw_text)
    card_name = extract_main_card_name(raw_lines, fallback_slug=slug_from_url(url))
    main_lines = crop_to_main_product_block(raw_lines, card_name)
    main_text = "\n".join(main_lines)

    family = infer_family(card_id, card_name)

    annual_fee = extract_annual_fee(main_lines)
    annual_fee_full = extract_annual_fee_full(main_lines)
    additional_card_fee = extract_additional_card_fee(main_lines)
    purchase_interest_rate = extract_purchase_interest_rate(main_lines)
    cash_interest_rate = extract_cash_interest_rate(main_lines)
    income_requirement = extract_income_requirement(main_lines)
    welcome_bonus_summary = extract_welcome_bonus_summary(main_lines)

    overview_text = extract_overview_block(main_lines, card_name)
    offer_text = extract_offer_block(main_lines)
    rewards_text = extract_rewards_block(main_lines, family)
    benefits_text = extract_benefits_block(main_lines, family)
    insurance_text = extract_insurance_block(main_lines)
    documents_text = extract_documents_block(main_lines)

    record = {
        "card_id": card_id,
        "card_name": card_name,
        "issuer": "CIBC",
        "country": "Canada",
        "network": infer_network(card_name, url),
        "card_type": "Credit Card",
        "card_family": family,
        "link": url,
        "annual_fee": annual_fee,
        "annual_fee_full": annual_fee_full,
        "additional_card_fee": additional_card_fee,
        "purchase_interest_rate": purchase_interest_rate,
        "cash_interest_rate": cash_interest_rate,
        "income_requirement": income_requirement,
        "welcome_bonus_summary": welcome_bonus_summary,
        "best_for": infer_best_for(family, main_text),
        "one_liner": infer_one_liner(family),
    }

    snippets = []
    for section_name, section_text in [
        ("overview", overview_text),
        ("welcome_bonus", offer_text),
        ("rewards", rewards_text),
        ("benefits", benefits_text),
        ("insurance", insurance_text),
        ("documents", documents_text),
        ("eligibility", income_requirement),
    ]:
        snippet = make_snippet(card_id, section_name, section_text)
        if snippet:
            snippets.append(snippet)

    return record, snippets


# =========================================================
# RUN
# =========================================================
def run():
    all_records = []
    all_snippets = []

    for card in CIBC_CARDS:
        print(f"Scraping: {card['card_id']}")
        try:
            record, snippets = scrape_cibc_card(card)
            all_records.append(record)
            all_snippets.extend(snippets)

            print("  card_name            :", record["card_name"])
            print("  annual_fee           :", record["annual_fee"])
            print("  income_requirement   :", record["income_requirement"])
            print("  welcome_bonus_summary:", record["welcome_bonus_summary"])
            print("  snippets             :", len(snippets))

        except Exception as e:
            print(f"  FAILED: {e}")

    cards_df = pd.DataFrame(all_records)
    cards_df.to_csv(CSV_PATH, index=False)

    with open(SNIPPETS_PATH, "w", encoding="utf-8") as f:
        for item in all_snippets:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print("\nSaved CSV  :", CSV_PATH)
    print("Saved JSONL:", SNIPPETS_PATH)

    return cards_df, all_snippets


if __name__ == "__main__":
    df, snippets = run()
    print("\nPreview:")
    print(df[[
        "card_id",
        "card_name",
        "annual_fee",
        "income_requirement",
        "welcome_bonus_summary"
    ]].to_string(index=False))

Scraping: cibc_aventura_visa_infinite_ca
  card_name            : CIBC Aventura® Visa Infinite Card
  annual_fee           : 139
  income_requirement   : $60,000 individual, or $100,000 household minimum annual income
  welcome_bonus_summary: Join and earn up to $1,400 in value
  snippets             : 7
Scraping: cibc_aeroplan_visa_infinite_ca
  card_name            : CIBC Aeroplan® Visa Infinite Card
  annual_fee           : 139
  income_requirement   : $60,000 individual, or $100,000 household minimum annual income
  welcome_bonus_summary: Apply and earn up to $1,200 in value, which includes up to 45,000 Aeroplan points.
  snippets             : 5
Scraping: cibc_aeroplan_visa_infinite_privilege_ca
  card_name            : CIBC Aeroplan® Visa Infinite Privilege Card
  annual_fee           : 599
  income_requirement   : $150,000 individual, or $200,000 household minimum annual income
  welcome_bonus_summary: Apply and earn up to $3,200 in value, which includes up to 100,000 Aeroplan p

In [41]:
cards_df.head()

,card_id,card_name,issuer,country,network,card_type,card_family,link,annual_fee,annual_fee_full,purchase_interest_rate,cash_interest_rate,income_requirement,welcome_bonus_summary,best_for,one_liner
0,cibc_aventura_visa_infinite_ca,,CIBC,Canada,Visa,Credit Card,aventura,https://www.cibc.com/en/personal-banking/credi...,,,,,,,travel,Travel rewards credit card designed for flexib...
1,cibc_aeroplan_visa_infinite_ca,,CIBC,Canada,Visa,Credit Card,aeroplan,https://www.cibc.com/en/personal-banking/credi...,,,,,,,"travel,air_travel",Travel credit card focused on Aeroplan earning...
2,cibc_aeroplan_visa_infinite_privilege_ca,,CIBC,Canada,Visa,Credit Card,aeroplan,https://www.cibc.com/en/personal-banking/credi...,,,,,,,"travel,air_travel",Travel credit card focused on Aeroplan earning...
3,cibc_aventura_gold_visa_ca,,CIBC,Canada,Visa,Credit Card,aventura,https://www.cibc.com/en/personal-banking/credi...,,,,,,,travel,Travel rewards credit card designed for flexib...
4,cibc_aventura_visa_ca,,CIBC,Canada,Visa,Credit Card,aventura,https://www.cibc.com/en/personal-banking/credi...,,,,,,,travel,Travel rewards credit card designed for flexib...
